In [2]:
!pip install python-dotenv 
!pip install openai

In [3]:
from dotenv import load_dotenv
import os
import sys 
sys.path.append('../..')
from dataset.vqav2 import VQADataset_json, VQADataset
import openai

load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY")) 

In [4]:
vqa1k = VQADataset_json(prompt='', \
                        image_dir_path="/home/david/Desktop/yuna/HPA/data/val2014", \
                        json_path="/home/david/Desktop/yuna/HPA/dataset/vqav2_1k_val.json") 
qids = [d['question_id'] for d in vqa1k]

In [5]:
dataset = VQADataset( image_dir_path="/home/david/Desktop/yuna/data/val2014", 
            question_path="/home/david/Desktop/yuna/data/v2_OpenEnded_mscoco_val2014_questions.json", 
            annotations_path="/home/david/Desktop/yuna/data/v2_mscoco_val2014_annotations.json", 
            prompt='') 

In [6]:
questions = [q for q in dataset.questions if q['question_id'] in qids]
questions[0]

{'image_id': 524577,
 'question': 'What number of clocks are on this tower?',
 'question_id': 524577006}

In [7]:
from typing import List, Dict, Any, Iterable

def est_tokens_for_question(q: str) -> int:
    # Conservative heuristic: input tokens
    return max(1, len(q) // 4) + 8

def est_output_tokens_per_item() -> int:
    # Your schema is large; be conservative.
    # Tune this after you run 1-2 batches and observe actual output sizes.
    return 180

def make_batches(items: List[Dict[str, Any]],
                 max_est_total_tokens: int = 18000,
                 hard_max_items: int = 60) -> Iterable[List[Dict[str, Any]]]:
    """
    items: [{"qid":..., "question":...}, ...]
    max_est_total_tokens: combined estimated input+output budget per request
    hard_max_items: absolute cap per batch to avoid huge outputs
    """
    batch = []
    cur = 0
    out_per = est_output_tokens_per_item()

    for it in items:
        q = it["question"]
        add = est_tokens_for_question(q) + out_per

        # start new batch if we'd exceed budget or item cap
        if batch and (cur + add > max_est_total_tokens or len(batch) >= hard_max_items):
            yield batch
            batch = []
            cur = 0

        batch.append(it)
        cur += add

    if batch:
        yield batch

# Example usage:
# for b in make_batches(all_items, max_est_total_tokens=18000, hard_max_items=60):
#     results = extract_semantics(b)

In [ ]:
SYSTEM_SEM = (
    """
    Extract VQA question semantics from text only; do not invent visual details.
    Return ONLY a JSON object {"results":[...]} matching the schema exactly; preserve input order.
    Decision rules:
    1) op priority:
    text(reading cue: written/word/number/say/spell/label/sign/license; brand/logo ONLY if implied visible/printed) >
    count(how many/number of) >
    spat(explicit left/right/top/bottom/foreground/background; where-questions; relations like behind/next to) >
    temp(how long/how old/aged/before/after) >
    cause(why/reason) >
    comp(comparative/superlative) >
    attr(color/material/type/age/duration/name/location asked) >
    exist(only existential presence: "is/are there", "any", "present") >
    ident(who/what/which entity is) >
    act(activity/doing) >
    know(world knowledge beyond simple visual attributes, when no text-reading cue) > other.
    2) If w=yesno AND op=exist => ans=bool, space=bin. Other yes/no use op based on content.
    3) attr consistency: color->ans=color,space=cat; mat->mat,cat; age->age,num; dur->dur,num;
    count->num,num; name->text,cat; loc->loc,cat; type/ident->cat,cat.
    4) txt=true iff op=text. know=true iff op=know.
    5) sp/rel/dx only if explicit in question.
    6) If w=where and op not in {text,know} => op=spat, attr=loc, ans=loc, space=cat.
    """ 
)

SYSTEM_CTL = (
    "Generate 4 control variants for each VQA question. "
    "Return ONLY a JSON object {\"results\":[...]}; preserve input order.\n"
    "Rules:\n"
    "1) deictic_removed: Remove deictic words (this/that/these/those/here/there) "
    "and spatial markers (left/right/top/bottom/foreground/background) while preserving grammar. "
    "Restructure the sentence if needed to remain fluent.\n"
    "2) object_removed: Replace the specific object noun with neutral placeholder 'object'. "
    "Do not replace pronouns or generic nouns already present.\n"
    "3) weaker_object: Replace the specific object noun with a broader hypernym one level up "
    "(dog->animal, car->vehicle, drink->beverage, shirt->clothing, apple->fruit, "
    "knife->utensil, sofa->furniture, bus->vehicle, cat->animal). "
    "If no clear hypernym exists, use 'object'.\n"
    "4) subject_ablated: Replace the subject/object noun phrase with 'the object', 'the thing', or 'it' "
    "while preserving grammatical structure. Do not introduce new entities not in the original question."
    "(dog->animal, car->vehicle, drink->beverage, shirt->clothing, apple->fruit, etc.).\n"
    "- subject_ablated: replace subject/object NP with 'the object' or 'it'; preserve syntax; do not add entities."
)

# ── Schemas ────────────────────────────────────────────────────────────────────

SEM_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "w":     {"type": "string", "enum": ["what","which","who","where","when","why","how","how_many","how_much","how_long","how_old","yesno","other"]},
                    "op":    {"type": "string", "enum": ["exist","ident","attr","count","act","spat","temp","text","know","cause","comp","other"]},
                    "p":     {"type": ["string","null"]},
                    "sub":   {"type": "string"},
                    "obj":   {"type": ["string","null"]},
                    "ent":   {"type": "string", "enum": ["person","animal","vehicle","food","product","place","text","object","other"]},
                    "attr":  {"type": ["string","null"], "enum": ["color","age","count","dur","mat","ident","type","name","loc","other",None]},
                    "ans":   {"type": "string", "enum": ["bool","num","dur","age","color","mat","loc","person","object","cat","text","open"]},
                    "space": {"type": "string", "enum": ["bin","num","cat","open"]},
                    "sp":    {"type": "array", "items": {"type": "string", "enum": ["left","right","top","bottom","front","back","center","foreground","background"]}},
                    "rel":   {"type": "array", "items": {"type": "string", "enum": ["on","in","next","behind","hold","wear","under","above","between","near"]}},
                    "dx":    {"type": "array", "items": {"type": "string", "enum": ["this","that","these","those","here","there"]}},
                    "neg":   {"type": "boolean"},
                    "know":  {"type": "boolean"},
                    "txt":   {"type": "boolean"}, 
                },
                "required": ["w","op","p","sub","obj","ent","attr","ans","space","sp","rel","dx","neg","know","txt","q"]
            }
        }
    },
    "required": ["results"],
    "additionalProperties": False
}

CTL_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "deictic_removed":   {"type": "string"},
                    "object_removed":    {"type": "string"},
                    "weaker_object":     {"type": "string"},
                    "subject_ablated":   {"type": "string"}
                },
                "required": ["deictic_removed","object_removed","weaker_object","subject_ablated"]
            }
        }
    },
    "required": ["results"],
    "additionalProperties": False
}


In [9]:
import json 
import time
from pathlib import Path  
from typing import List, Dict, Any

MODEL = "gpt-4.1-mini-2025-04-14"  # use a snapshot for reproducibility (if available in your account) 

# ── Core API call ──────────────────────────────────────────────────────────────

def call_api(system: str, schema: dict[str, Any], name: str, questions: list[str], retries=3) -> list[dict]:
    payload = json.dumps(questions, ensure_ascii=False)
    for attempt in range(retries):
        try:
            resp = client.responses.create(
                model=MODEL,
                input=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": payload},
                ],
                text={"format": {"type": "json_schema", "name": name, "schema": schema, "strict": True}},
                temperature=0,
            )
            data = json.loads(resp.output_text)
            results = data.get("results", [])
            if len(results) != len(questions):
                raise ValueError(f"Length mismatch: got {len(results)}, expected {len(questions)}")
            return results
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = 2 ** attempt
            print(f"  Retry {attempt+1} after {wait}s: {e}")
            time.sleep(wait) 
            

In [21]:
BATCH_SIZE = 50 

def process(questions, mode='CTL', output_path: str = "vqa_extracted.jsonl", batch_size: int = BATCH_SIZE):
    output = []
    batches = make_batches(questions, hard_max_items=batch_size) 
    if mode == 'CTL': 
        system = SYSTEM_CTL
        schema= CTL_SCHEMA 
    else: 
        system = SYSTEM_SEM 
        schema= SEM_SCHEMA  

    # Open the file in append mode ('a')
    with open(output_path, "a", encoding="utf-8") as f:
        for batch in batches:  
            # Extract questions and call API
            questions_text = [q['question'] for q in batch]
            ctl_results = call_api(system, schema, "vqa_ctl_v1", questions_text) 
            
            # Merge results
            current_batch_processed = [
                {**s, **c} for s, c in zip(batch, ctl_results)
            ]
            
            # Write only the NEW items to the file immediately
            for item in current_batch_processed:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
            
            output.extend(current_batch_processed)
            print(f"Saved {len(output)} total items (Batch size: {len(batch)})")
            
    return output 

In [24]:
savefile = "./dataset/vqa1k_control.json" 
savefile = "./dataset/vqa1k_semantics.json" 
output = process(questions, mode='SEM', output_path=savefile)  

Saved 50 total items (Batch size: 50)
Saved 100 total items (Batch size: 50)
Saved 150 total items (Batch size: 50)
Saved 200 total items (Batch size: 50)
Saved 250 total items (Batch size: 50)
Saved 300 total items (Batch size: 50)
Saved 350 total items (Batch size: 50)
Saved 400 total items (Batch size: 50)
Saved 450 total items (Batch size: 50)
Saved 500 total items (Batch size: 50)
Saved 550 total items (Batch size: 50)
  Retry 1 after 1s: Length mismatch: got 49, expected 50
Saved 600 total items (Batch size: 50)
Saved 650 total items (Batch size: 50)
Saved 700 total items (Batch size: 50)
Saved 750 total items (Batch size: 50)
Saved 800 total items (Batch size: 50)
Saved 850 total items (Batch size: 50)
Saved 900 total items (Batch size: 50)
Saved 950 total items (Batch size: 50)
Saved 1000 total items (Batch size: 50)


In [41]:
from datasets import load_dataset

textvqa = load_dataset("lmms-lab/textvqa", split="validation")  
okvqa = load_dataset("lmms-lab/OK-VQA", split="val2014") 

Generating val2014 split: 100%|██████████| 5046/5046 [00:01<00:00, 3682.05 examples/s]
